# Gold Layer - Merit Order Cost Analysis

## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Configuing Paths

In [0]:
SILVER_TABLE = "rf_assessment.silver.merit_order_data"

# Load silver data
df_silver = spark.table(SILVER_TABLE)

print("📊 Silver Layer Data Loaded")
print(f"Total Records: {df_silver.count():,}")
print(f"\nDate Range: {df_silver.agg(F.min('effective_date')).collect()[0][0]} to {df_silver.agg(F.max('effective_date')).collect()[0][0]}")
print(f"\nMonths Available: {df_silver.select('month', 'year').distinct().count()}")

display(df_silver.groupBy('month', 'year').count().orderBy('year', 'month'))

## Load Silver Layer Data

In [0]:
# Calculate plant-wise fuel cost statistics across months
plant_cost_variation = df_silver.groupBy('plant_name', 'fuel_type_clean').agg(
    F.min('fuel_cost').alias('min_fuel_cost'),
    F.max('fuel_cost').alias('max_fuel_cost'),
    F.avg('fuel_cost').alias('avg_fuel_cost'),
    F.stddev('fuel_cost').alias('std_fuel_cost'),
    F.count('*').alias('observation_count')
).withColumn(
    'cost_variation_range', F.col('max_fuel_cost') - F.col('min_fuel_cost')
).withColumn(
    'cost_variation_pct', 
    F.when(F.col('min_fuel_cost') > 0, 
           ((F.col('max_fuel_cost') - F.col('min_fuel_cost')) / F.col('min_fuel_cost') * 100))
    .otherwise(None)
).orderBy(F.desc('cost_variation_range'))

print("📈 Top 10 Plants with Highest Fuel Cost Variation:")
display(plant_cost_variation.limit(10))

In [0]:
# Save as gold table
plant_cost_variation.write.mode('overwrite').saveAsTable('rf_assessment.gold.plant_cost_variation')
print("✅ Saved to: rf_assessment.gold.plant_cost_variation")

## Analysis 1: Plant-wise Fuel Cost Variation

Identify which plants have the highest fuel cost fluctuations across the 4-month period.

In [0]:
# Aggregate costs by plant and month
monthly_plant_costs = df_silver.groupBy('plant_name', 'fuel_type_clean', 'month', 'year').agg(
    F.avg('fuel_cost').alias('avg_fuel_cost'),
    F.avg('specific_cost').alias('avg_specific_cost'),
    F.count('*').alias('records_count')
).orderBy('plant_name', 'year', 'month')

print("📊 Monthly Plant Costs Sample:")
display(monthly_plant_costs.limit(20))

In [0]:
# Save as gold table
monthly_plant_costs.write.mode('overwrite').saveAsTable('rf_assessment.gold.monthly_plant_costs')
print("✅ Saved to: rf_assessment.gold.monthly_plant_costs")

## Analysis 2: Monthly Trend by Plant

Track how each plant's costs change month-over-month.

In [0]:
# Calculate quartiles and IQR for each fuel type
from pyspark.sql.functions import expr

fuel_type_stats = df_silver.groupBy('fuel_type_clean').agg(
    F.expr('percentile(fuel_cost, 0.25)').alias('q1'),
    F.expr('percentile(fuel_cost, 0.75)').alias('q3'),
    F.avg('fuel_cost').alias('avg_fuel_cost')
).withColumn('iqr', F.col('q3') - F.col('q1'))

print("Calculated Q1, Q3, and IQR for each fuel type")
display(fuel_type_stats)

In [0]:
# Join with main data and identify outliers (values outside 1.5 * IQR from quartiles)
df_with_stats = df_silver.join(fuel_type_stats, 'fuel_type_clean')

outliers = df_with_stats.filter(
    (F.col('fuel_cost') < (F.col('q1') - 1.5 * F.col('iqr'))) |
    (F.col('fuel_cost') > (F.col('q3') + 1.5 * F.col('iqr')))
).select(
    'plant_name', 'fuel_type_clean', 'effective_date', 'month',
    'fuel_cost', 'specific_cost', 'avg_fuel_cost'
).withColumn(
    'deviation_pct',
    ((F.col('fuel_cost') - F.col('avg_fuel_cost')) / F.col('avg_fuel_cost') * 100)
).orderBy(F.desc(F.abs(F.col('deviation_pct'))))

print(f"🚨 Outliers Detected: {outliers.count()} records")
print("\nTop 15 Cost Outliers:")
display(outliers.limit(15))

In [0]:
# Save outliers to gold table
outliers.write.mode('overwrite').saveAsTable('rf_assessment.gold.cost_outliers')
print("✅ Saved to: rf_assessment.gold.cost_outliers")

## Analysis 3: Outlier Detection

Identify anomalous cost values using the IQR (Interquartile Range) method.

In [0]:
# Analyze fuel type costs by month
fuel_type_analysis = df_silver.groupBy('fuel_type_clean', 'month').agg(
    F.avg('fuel_cost').alias('avg_fuel_cost'),
    F.min('fuel_cost').alias('min_fuel_cost'),
    F.max('fuel_cost').alias('max_fuel_cost'),
    F.count('*').alias('plant_count')
).orderBy('fuel_type_clean', 'month')

print("⛽ Fuel Type Analysis by Month:")
display(fuel_type_analysis)

In [0]:
# Overall fuel type cost comparison (aggregated across all months)
fuel_summary = df_silver.groupBy('fuel_type_clean').agg(
    F.avg('fuel_cost').alias('avg_fuel_cost'),
    F.avg('specific_cost').alias('avg_specific_cost'),
    F.countDistinct('plant_name').alias('unique_plants'),
    F.count('*').alias('total_records')
).orderBy(F.desc('avg_fuel_cost'))

print("⛽ Overall Fuel Type Cost Comparison:")
display(fuel_summary)

In [0]:
# Save fuel type analysis tables to gold layer
fuel_type_analysis.write.mode('overwrite').saveAsTable('rf_assessment.gold.fuel_type_monthly_analysis')
fuel_summary.write.mode('overwrite').saveAsTable('rf_assessment.gold.fuel_type_summary')
print("✅ Saved fuel type analysis tables")

## Analysis 4: Fuel Type Impact Analysis

Compare cost patterns across different fuel types.

In [0]:
# Visualization 1: Top 15 plants with highest cost variation
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Get top 15 plants with highest variation
top_plants_pdf = plant_cost_variation.limit(15).toPandas()

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(top_plants_pdf['plant_name'], top_plants_pdf['cost_variation_range'], color='steelblue')

# Color code by fuel type
colors = {'COAL': 'brown', 'GAS': 'orange', 'RLNG': 'green', 'RFO': 'red', 'NUCLEAR': 'purple', 'HYDEL': 'blue'}
for i, (plant, fuel_type) in enumerate(zip(top_plants_pdf['plant_name'], top_plants_pdf['fuel_type_clean'])):
    bars[i].set_color(colors.get(fuel_type, 'gray'))

ax.set_xlabel('Fuel Cost Variation Range (Rs/kWh)', fontsize=12, fontweight='bold')
ax.set_ylabel('Plant Name', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Plants with Highest Fuel Cost Variation\n(May - August 2026)', fontsize=14, fontweight='bold')

# Add value labels
for i, v in enumerate(top_plants_pdf['cost_variation_range']):
    ax.text(v + 0.5, i, f'{v:.2f}', va='center', fontsize=9)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=fuel) for fuel, color in colors.items()]
ax.legend(handles=legend_elements, title='Fuel Type', loc='lower right')

plt.tight_layout()
plt.show()

print("✅ Chart 1: Cost Variation by Plant")

## Data Visualizations

Visual representations of the key insights discovered in the analysis.

In [0]:
# Visualization 2: Monthly fuel cost trends for top 10 variable plants
top_10_plants = [row['plant_name'] for row in plant_cost_variation.limit(10).collect()]

trend_data = df_silver.filter(F.col('plant_name').isin(top_10_plants)).groupBy('plant_name', 'month').agg(
    F.avg('fuel_cost').alias('avg_fuel_cost')
).toPandas()

fig, ax = plt.subplots(figsize=(14, 8))

for plant in top_10_plants:
    plant_data = trend_data[trend_data['plant_name'] == plant].sort_values('month')
    ax.plot(plant_data['month'], plant_data['avg_fuel_cost'], marker='o', linewidth=2, label=plant)

ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Fuel Cost (Rs/kWh)', fontsize=12, fontweight='bold')
ax.set_title('Monthly Fuel Cost Trends - Top 10 Most Variable Plants', fontsize=14, fontweight='bold')
ax.set_xticks([5, 6, 7, 8])
ax.set_xticklabels(['May', 'June', 'July', 'August'])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Chart 2: Monthly Trends")

In [0]:
# Visualization 3: Fuel type cost comparison
fuel_summary_pdf = fuel_summary.toPandas()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Average costs by fuel type
colors_list = [colors.get(ft, 'gray') for ft in fuel_summary_pdf['fuel_type_clean']]
ax1.bar(fuel_summary_pdf['fuel_type_clean'], fuel_summary_pdf['avg_fuel_cost'], color=colors_list, alpha=0.7)
ax1.bar(fuel_summary_pdf['fuel_type_clean'], fuel_summary_pdf['avg_specific_cost'], 
        bottom=fuel_summary_pdf['avg_fuel_cost'], color=colors_list, alpha=0.4, label='Other Costs')

ax1.set_xlabel('Fuel Type', fontsize=12, fontweight='bold')
ax1.set_ylabel('Average Cost (Rs/kWh)', fontsize=12, fontweight='bold')
ax1.set_title('Average Cost by Fuel Type\n(Fuel Cost + Other Costs)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for i, (fuel, specific) in enumerate(zip(fuel_summary_pdf['avg_fuel_cost'], fuel_summary_pdf['avg_specific_cost'])):
    ax1.text(i, specific + fuel + 1, f'{specific + fuel:.1f}', ha='center', fontsize=9, fontweight='bold')

# Chart 2: Plant count by fuel type
ax2.pie(fuel_summary_pdf['unique_plants'], labels=fuel_summary_pdf['fuel_type_clean'], 
        autopct='%1.1f%%', colors=colors_list, startangle=90)
ax2.set_title('Distribution of Plants by Fuel Type', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Chart 3: Fuel Type Analysis")

In [0]:
# Visualization 4: Outliers analysis
if outliers.count() > 0:
    outliers_pdf = outliers.limit(30).toPandas()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Chart 1: Outliers by fuel type
    outlier_by_fuel = outliers.groupBy('fuel_type_clean').count().toPandas()
    colors_list = [colors.get(ft, 'gray') for ft in outlier_by_fuel['fuel_type_clean']]
    ax1.bar(outlier_by_fuel['fuel_type_clean'], outlier_by_fuel['count'], color=colors_list, alpha=0.7)
    ax1.set_xlabel('Fuel Type', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Outlier Records', fontsize=12, fontweight='bold')
    ax1.set_title('Outlier Distribution by Fuel Type', fontsize=13, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, v in enumerate(outlier_by_fuel['count']):
        ax1.text(i, v + 1, str(v), ha='center', fontweight='bold')
    
    # Chart 2: Top plants with outliers
    outlier_by_plant = outliers.groupBy('plant_name').count().orderBy(F.desc('count')).limit(10).toPandas()
    ax2.barh(outlier_by_plant['plant_name'], outlier_by_plant['count'], color='coral')
    ax2.set_xlabel('Number of Outlier Records', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Plant Name', fontsize=12, fontweight='bold')
    ax2.set_title('Top 10 Plants with Most Outliers', fontsize=13, fontweight='bold')
    
    # Add value labels
    for i, v in enumerate(outlier_by_plant['count']):
        ax2.text(v + 0.2, i, str(v), va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Chart 4: Outliers Analysis")
else:
    print("ℹ️ No outliers detected in the data")

In [0]:
# Calculate overall dataset statistics
total_plants = df_silver.select('plant_name').distinct().count()
total_fuel_types = df_silver.select('fuel_type_clean').distinct().count()
avg_fuel_cost_overall = df_silver.agg(F.avg('fuel_cost')).collect()[0][0]

print("="*80)
print("KEY FINDINGS SUMMARY - MERIT ORDER COST ANALYSIS (May - August 2026)")
print("="*80)
print(f"\n📊 DATASET OVERVIEW:")
print(f"  • Total Plants Analyzed: {total_plants}")
print(f"  • Fuel Types: {total_fuel_types}")
print(f"  • Average Fuel Cost: Rs {avg_fuel_cost_overall:.2f}/kWh")
print(f"  • Total Observations: 861 records across 4 months")

In [0]:
# Display top 5 plants with highest cost variation
top_5_variable = plant_cost_variation.limit(5).collect()

print("\n📈 TOP 5 PLANTS WITH HIGHEST COST VARIATION:")
for i, row in enumerate(top_5_variable, 1):
    print(f"  {i}. {row['plant_name']} ({row['fuel_type_clean']})")
    print(f"     - Variation: Rs {row['cost_variation_range']:.2f}/kWh ({row['cost_variation_pct']:.1f}%)")
    print(f"     - Range: Rs {row['min_fuel_cost']:.2f} - Rs {row['max_fuel_cost']:.2f}/kWh")

In [0]:
# Display fuel type cost insights
print("\n⛽ FUEL TYPE INSIGHTS:")
fuel_costs = fuel_summary.collect()
for row in fuel_costs:
    print(f"  • {row['fuel_type_clean']}: Avg Rs {row['avg_fuel_cost']:.2f}/kWh ({row['unique_plants']} plants)")

In [0]:
# Display outlier detection results
outlier_count = outliers.count()

if outlier_count > 0:
    print(f"\n🚨 OUTLIERS DETECTED: {outlier_count} records")
    top_outlier = outliers.first()
    print(f"  • Most Extreme: {top_outlier['plant_name']} ({top_outlier['fuel_type_clean']})")
    print(f"    - Deviation: {top_outlier['deviation_pct']:.1f}% from average")
    print(f"    - Cost: Rs {top_outlier['fuel_cost']:.2f}/kWh vs Avg Rs {top_outlier['avg_fuel_cost']:.2f}/kWh")
else:
    print("\nℹ️ No outliers detected in the dataset")

In [0]:
# List all gold layer tables created
print(f"\n✅ GOLD LAYER TABLES CREATED:")
print(f"  • rf_assessment.gold.plant_cost_variation")
print(f"  • rf_assessment.gold.monthly_plant_costs")
print(f"  • rf_assessment.gold.cost_outliers")
print(f"  • rf_assessment.gold.fuel_type_monthly_analysis")
print(f"  • rf_assessment.gold.fuel_type_summary")
print("="*80)

## Executive Summary

In [0]:
# Key findings from the merit order cost analysis across May - August 2026

print("="*80)
print("KEY FINDINGS FROM MERIT ORDER COST ANALYSIS (May - August 2026)")
print("="*80)

# Dataset Overview
total_plants = df_silver.select('plant_name').distinct().count()
total_fuel_types = df_silver.select('fuel_type_clean').distinct().count()
avg_fuel_cost_overall = df_silver.agg(F.avg('fuel_cost')).collect()[0][0]
total_records = df_silver.count()

print(f"\n📊 DATASET OVERVIEW:")
print(f"  • Total Plants Analyzed: {total_plants}")
print(f"  • Fuel Types: {total_fuel_types}")
print(f"  • Average Fuel Cost: Rs {avg_fuel_cost_overall:.2f}/kWh")
print(f"  • Total Observations: {total_records} records across 4 months")

# Top Variable Plants
top_5_variable = plant_cost_variation.limit(5).collect()

print("\n📈 TOP 5 PLANTS WITH HIGHEST COST VARIATION:")
for i, row in enumerate(top_5_variable, 1):
    print(f"  {i}. {row['plant_name']} ({row['fuel_type_clean']})")
    print(f"     - Variation: Rs {row['cost_variation_range']:.2f}/kWh ({row['cost_variation_pct']:.1f}%)")
    print(f"     - Range: Rs {row['min_fuel_cost']:.2f} - Rs {row['max_fuel_cost']:.2f}/kWh")

# Fuel Type Insights
print("\n⛽ FUEL TYPE COST COMPARISON:")
fuel_costs = fuel_summary.orderBy(F.desc('avg_fuel_cost')).collect()
for row in fuel_costs:
    print(f"  • {row['fuel_type_clean']}: Avg Rs {row['avg_fuel_cost']:.2f}/kWh ({row['unique_plants']} plants)")

# Outlier Detection Results
outlier_count = outliers.count()

if outlier_count > 0:
    print(f"\n🚨 OUTLIERS DETECTED: {outlier_count} records")
    top_outlier = outliers.first()
    print(f"  • Most Extreme: {top_outlier['plant_name']} ({top_outlier['fuel_type_clean']})")
    print(f"    - Deviation: {top_outlier['deviation_pct']:.1f}% from average")
    print(f"    - Cost: Rs {top_outlier['fuel_cost']:.2f}/kWh vs Avg Rs {top_outlier['avg_fuel_cost']:.2f}/kWh")
else:
    print("\nℹ️ No significant outliers detected in the dataset")

# Gold Layer Tables Summary
print(f"\n✅ GOLD LAYER TABLES CREATED:")
print(f"  • rf_assessment.gold.plant_cost_variation")
print(f"  • rf_assessment.gold.monthly_plant_costs")
print(f"  • rf_assessment.gold.cost_outliers")
print(f"  • rf_assessment.gold.fuel_type_monthly_analysis")
print(f"  • rf_assessment.gold.fuel_type_summary")
print("="*80)